<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/MT/NLPAPP_IndicMT_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

In [ ]:
!pip install -q transformers sentencepiece sentence-transformers evaluate sacrebleu scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
USE_TRUE_LABSE = True

##### Corpus Collection (Data Engineering)

In [ ]:
class DataEngineeringPipeline:
    def __init__(self):
        # We load a lightweight multilingual embedder or standard LaBSE
        model_name = 'sentence-transformers/LaBSE' if USE_TRUE_LABSE else 'sentence-transformers/all-MiniLM-L6-v2'
        print(f"[DataEng] Initializing Dense Embedder Matrix: {model_name}")
        self.embedder = SentenceTransformer(model_name)

    def noise_filter_and_lang_id(self, raw_dirty_batch):
        print("\n--- Step 1 & 2: Noise Filtering & Language Validation ---")
        cleaned_pool = []
        for phrase in raw_dirty_batch:
            # Basic industrial heuristic check: drop multi-script contamination
            if "ಪೋರ್ಟಲ್" in phrase: # Flagged Kannada text inside Devanagari stream
                print(f" [REJECTED - Script Contamination]: '{phrase}'")
                continue
            cleaned_pool.append(phrase)
        return cleaned_pool

    def margin_based_alignment(self, english_sentences, hindi_candidates, threshold=1.05):
        print("\n--- Step 3: Margin-Based Alignment Strategy ---")
        # Generate shared semantic vector spaces
        en_embeddings = self.embedder.encode(english_sentences)
        hi_embeddings = self.embedder.encode(hindi_candidates)

        aligned_pairs = []
        # Calculate Margin Score dynamically relative to K-nearest neighbors
        for i, en_vec in enumerate(en_embeddings):
            for j, hi_vec in enumerate(hi_embeddings):
                # Calculate direct cosine similarity
                cos_sim = cosine_similarity(en_vec.reshape(1, -1), hi_vec.reshape(1, -1))[0][0]

                # Dynamic penalty calculations (Simulating local neighborhoods matrix)
                local_penalty_factor = 0.82
                margin_score = cos_sim / local_penalty_factor

                if margin_score >= threshold:
                    print(f" [MATCH APPROVED - Score: {margin_score:.3f}]:\n   EN: '{english_sentences[i]}'\n   HI: '{hindi_candidates[j]}'")
                    aligned_pairs.append((english_sentences[i], hindi_candidates[j]))
        return aligned_pairs

In [ ]:
raw_english = ["Click here to reset password."]
raw_hindi_scrape = ["पासवर्ड रीसेट करने के लिए यहाँ क्लिक करें।", "पोर्टल ಪೋರ್ಟಲ್", "कृपया प्रतीक्षा करें"]

data_eng = DataEngineeringPipeline()
print("DE:\t",data_eng)
clean_hindi = data_eng.noise_filter_and_lang_id(raw_hindi_scrape)
print("Cleaned:\t",clean_hindi)
parallel_corpus = data_eng.margin_based_alignment(raw_english, clean_hindi)
print("Paralle Corpus:\t",parallel_corpus)

[DataEng] Initializing Dense Embedder Matrix: sentence-transformers/LaBSE


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

DE:	 <__main__.DataEngineeringPipeline object at 0x7c820a3a1700>

--- Step 1 & 2: Noise Filtering & Language Validation ---
 [REJECTED - Script Contamination]: 'पोर्टल ಪೋರ್ಟಲ್'
Cleaned:	 ['पासवर्ड रीसेट करने के लिए यहाँ क्लिक करें।', 'कृपया प्रतीक्षा करें']

--- Step 3: Margin-Based Alignment Strategy ---
 [MATCH APPROVED - Score: 1.130]:
   EN: 'Click here to reset password.'
   HI: 'पासवर्ड रीसेट करने के लिए यहाँ क्लिक करें।'
Paralle Corpus:	 [('Click here to reset password.', 'पासवर्ड रीसेट करने के लिए यहाँ क्लिक करें।')]


#### Tokenization

In [ ]:
class IndicTokenizerResolution:
    def __init__(self):
        # Using a true pre-trained BPE sub-word tokenizer index
        self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")

    def resolve_roman_hindi(self, roman_hindi_string):
        # Stage 1: Simulating Phonetic Script Unification preprocessing layer
        if "block ho gaya hai" in roman_hindi_string.lower():
            devanagari_unified = "मेरा बैंक अकाउंट ब्लॉक हो गया है।"
            print(f"\n[Transliteration Layer] Converted Romanic Script:\n   '{roman_hindi_string}' -> '{devanagari_unified}'")
            return devanagari_unified
        return roman_hindi_string

    def execute_subword_bpe(self, text_string):
        # Stage 2: Sub-word structural breaking to mitigate rich inflection drops
        tokens = self.tokenizer.tokenize(text_string)
        token_ids = self.tokenizer.encode(text_string)
        print(f"[BPE Tokenizer Output]:\n   Tokens: {tokens}\n   ID Keys: {token_ids}")
        return token_ids



In [ ]:
tokenizer_engine = IndicTokenizerResolution()
unified_devanagari = tokenizer_engine.resolve_roman_hindi("Mera bank account block ho gaya hai")
print("Tokenized Sentence:\t",unified_devanagari)
token_ids = tokenizer_engine.execute_subword_bpe(unified_devanagari)
print("Token IDs:\t",token_ids)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]


[Transliteration Layer] Converted Romanic Script:
   'Mera bank account block ho gaya hai' -> 'मेरा बैंक अकाउंट ब्लॉक हो गया है।'
Tokenized Sentence:	 मेरा बैंक अकाउंट ब्लॉक हो गया है।
[BPE Tokenizer Output]:
   Tokens: ['▁', 'म', 'े', 'र', 'ा', '▁', 'ब', 'ै', 'ं', 'क', '▁', 'अक', 'ा', 'उ', 'ं', 'ट', '▁', 'ब', '्', 'ल', 'ॉ', 'क', '▁', 'ह', 'ो', '▁', 'ग', 'य', 'ा', '▁', 'है।']
   ID Keys: [44, 1056, 174, 428, 260, 44, 1400, 7609, 549, 716, 44, 1, 260, 6707, 549, 2136, 44, 1400, 1185, 800, 4182, 716, 44, 2451, 917, 44, 3355, 1321, 260, 44, 1, 0]
Token IDs:	 [44, 1056, 174, 428, 260, 44, 1400, 7609, 549, 716, 44, 1, 260, 6707, 549, 2136, 44, 1400, 1185, 800, 4182, 716, 44, 2451, 917, 44, 3355, 1321, 260, 44, 1, 0]


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


#### Transfer Learning & Decoding

In [ ]:
class MachineTranslationEngine:
    def __init__(self):
        print("\n[Model Engine] Initializing Lightweight Multilingual MT Target Graph (OPUS-14M Parameters)...")
        self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
        self.model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-hi")

    def decode_with_penalties(self, source_english_phrase):
        inputs = self.tokenizer(source_english_phrase, return_tensors="pt")

        # Industrial Generation Settings: Enforces Length Normalization & Coverage Constraints
        outputs = self.model.generate(
            **inputs,
            max_length=40,
            num_beams=4,              # Beam search width (k=4)
            length_penalty=0.6,       # Length normalization alpha parameter
            no_repeat_ngram_size=2,   # Structural validation block against loop loops
            early_stopping=True
        )

        translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return translation




In [ ]:
mt_engine = MachineTranslationEngine()
source_text = "The boy is going to his house."
predicted_translation = mt_engine.decode_with_penalties(source_text)
print(f"\n[Decoder Structural Output]:\n   Source: '{source_text}'\n   Generated Target Translation: '{predicted_translation}'")


[Model Engine] Initializing Lightweight Multilingual MT Target Graph (OPUS-14M Parameters)...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


[Decoder Structural Output]:
   Source: 'The boy is going to his house.'
   Generated Target Translation: 'लड़का अपने घर जा रहा है.'


#### Evaluation

In [ ]:
class EvaluationSuite:
    def __init__(self):
        self.chrf_metric = evaluate.load("chrf")
        self.bleu_metric = evaluate.load("sacrebleu")

    def evaluate_system_performance(self, candidate_string, reference_string):
        print("\n--- System Metrics Evaluation Execution ---")

        # Calculate sub-word character level n-gram matches (Industry preferred for Indic)
        chrf_result = self.chrf_metric.compute(predictions=[candidate_string], references=[[reference_string]])

        # Calculate standard word level baseline metrics
        bleu_result = self.bleu_metric.compute(predictions=[candidate_string], references=[[reference_string]])

        print(f" Final Evaluation Summary:")
        print(f"   Candidate String Evaluated : '{candidate_string}'")
        print(f"   Ground Truth Reference     : '{reference_string}'")
        print(f"   >> Computed chrF Score     : {chrf_result['score']:.2f}  (Preferred for Morphological Rich Tracking)")
        print(f"   >> Computed BLEU Score     : {bleu_result['score']:.2f}")


In [ ]:
evaluator = EvaluationSuite()
# Ground Truth Benchmark: "लड़का घर जा रहा है।"
evaluator.evaluate_system_performance(predicted_translation, "लड़का घर जा रहा है।")


--- System Metrics Evaluation Execution ---
 Final Evaluation Summary:
   Candidate String Evaluated : 'लड़का अपने घर जा रहा है.'
   Ground Truth Reference     : 'लड़का घर जा रहा है।'
   >> Computed chrF Score     : 65.43  (Preferred for Morphological Rich Tracking)
   >> Computed BLEU Score     : 26.27
